In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def resolve_results_dir() -> Path:
    """Prefer ./results when the kernel cwd is `notebooks/`; also try repo-relative paths."""
    for candidate in (
        Path("results"),
        Path("notebooks/results"),
        Path.cwd() / "results",
    ):
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find a results folder. Run this notebook with cwd = `notebooks/` "
        "or create `./results` next to this notebook."
    )


def is_under_crypto_strategy(results: Path, fp: Path) -> bool:
    rel = fp.relative_to(results)
    return bool(rel.parts) and rel.parts[0].endswith("_crypto")


RESULTS = resolve_results_dir()
print(f"Results directory: {RESULTS}\n")

all_csv = sorted(RESULTS.rglob("*.csv"))
csv_files = [fp for fp in all_csv if is_under_crypto_strategy(RESULTS, fp)]
print(f"Found {len(csv_files)} crypto CSV file(s) (of {len(all_csv)} total).\n")

merics_parts: list[pd.DataFrame] = []
trade_inventory: list[dict] = []

for fp in csv_files:
    rel = fp.relative_to(RESULTS)
    parts = rel.parts
    strategy = symbol = timeframe = ""
    if len(parts) >= 4:
        strategy, symbol, timeframe = parts[0], parts[1], parts[2]
    elif len(parts) >= 2:
        strategy = parts[0]

    try:
        df = pd.read_csv(fp)
    except Exception as exc:
        print(f"[skip] {rel} — {exc}")
        continue

    low = fp.name.lower()
    if low == "merics.csv":
        extra = pd.DataFrame(
            {
                "strategy": [strategy] * len(df),
                "symbol": [symbol] * len(df),
                "timeframe": [timeframe] * len(df),
                "rel_path": [str(rel)] * len(df),
            }
        )
        merics_parts.append(pd.concat([extra.reset_index(drop=True), df.reset_index(drop=True)], axis=1))
    elif low == "trades.csv":
        trade_inventory.append(
            {
                "strategy": strategy,
                "symbol": symbol,
                "timeframe": timeframe,
                "rows": len(df),
                "rel_path": str(rel),
            }
        )

display(Markdown("### Combined metrics (`merics.csv`) — crypto only"))
if merics_parts:
    all_metrics = pd.concat(merics_parts, ignore_index=True)
    front = ["strategy", "symbol", "timeframe", "rel_path"]
    rest = [c for c in all_metrics.columns if c not in front]
    all_metrics = all_metrics[front + sorted(rest, key=str.lower)]
    display(
        all_metrics.sort_values(["strategy", "symbol", "timeframe"]).reset_index(drop=True)
    )
else:
    display(Markdown("_No `merics.csv` files found under `*_crypto/`._"))

display(Markdown("### Trades files (`trades.csv`) — crypto only"))
if trade_inventory:
    display(
        pd.DataFrame(trade_inventory).sort_values(
            ["strategy", "symbol", "timeframe"]
        ).reset_index(drop=True)
    )
else:
    display(Markdown("_No `trades.csv` files found under `*_crypto/`._"))

display(Markdown("### Every crypto CSV path (relative to results root)"))
for fp in csv_files:
    print(fp.relative_to(RESULTS))

Results directory: D:\bot\ema-1d trend\notebooks\results

Found 28 crypto CSV file(s) (of 80 total).



### Combined metrics (`merics.csv`) — crypto only

,strategy,symbol,timeframe,rel_path,avg_pnl,end_balance,max_drawdown_%,net_pnl,profit_factor,return_%,start_balance,trades,win_rate_%
0,strategy03_crypto,ADAUSD,M5,strategy03_crypto\ADAUSD\M5\merics.csv,27.53,63183.61,14.23,53183.61,1.199,531.84,10000.0,1932,55.02
1,strategy03_crypto,AVEUSD,M5,strategy03_crypto\AVEUSD\M5\merics.csv,104.18,246901.65,17.31,236901.65,1.264,2369.02,10000.0,2274,57.30
2,strategy03_crypto,AVXUSD,M5,strategy03_crypto\AVXUSD\M5\merics.csv,12.60,27723.15,24.51,17723.15,1.126,177.23,10000.0,1407,53.87
3,strategy03_crypto,BCHUSD,M5,strategy03_crypto\BCHUSD\M5\merics.csv,57.35,131232.31,17.34,121232.31,1.226,1212.32,10000.0,2114,56.34
4,strategy03_crypto,BNBUSD,M5,strategy03_crypto\BNBUSD\M5\merics.csv,234.68,584273.18,12.82,574273.18,1.334,5742.73,10000.0,2447,58.56
5,strategy03_crypto,BTCUSD,M5,strategy03_crypto\BTCUSD\M5\merics.csv,425.89,1007862.38,12.64,997862.38,1.506,9978.62,10000.0,2343,60.09
6,strategy03_crypto,DOGEUSD,M5,strategy03_crypto\DOGEUSD\M5\merics.csv,22.40,39516.95,13.23,29516.95,1.192,295.17,10000.0,1318,55.46
7,strategy03_crypto,ETHUSD,M5,strategy03_crypto\ETHUSD\M5\merics.csv,269.26,587290.20,13.79,577290.20,1.446,5772.90,10000.0,2144,59.75
8,strategy03_crypto,LNKUSD,M5,strategy03_crypto\LNKUSD\M5\merics.csv,4.79,15964.21,37.45,5964.21,1.055,59.64,10000.0,1245,52.13
9,strategy03_crypto,LTCUSD,M5,strategy03_crypto\LTCUSD\M5\merics.csv,4.39,17572.80,36.53,7572.80,1.056,75.73,10000.0,1725,51.88


### Trades files (`trades.csv`) — crypto only

,strategy,symbol,timeframe,rows,rel_path
0,strategy03_crypto,ADAUSD,M5,1932,strategy03_crypto\ADAUSD\M5\trades.csv
1,strategy03_crypto,AVEUSD,M5,2274,strategy03_crypto\AVEUSD\M5\trades.csv
2,strategy03_crypto,AVXUSD,M5,1407,strategy03_crypto\AVXUSD\M5\trades.csv
3,strategy03_crypto,BCHUSD,M5,2114,strategy03_crypto\BCHUSD\M5\trades.csv
4,strategy03_crypto,BNBUSD,M5,2447,strategy03_crypto\BNBUSD\M5\trades.csv
5,strategy03_crypto,BTCUSD,M5,2343,strategy03_crypto\BTCUSD\M5\trades.csv
6,strategy03_crypto,DOGEUSD,M5,1318,strategy03_crypto\DOGEUSD\M5\trades.csv
7,strategy03_crypto,ETHUSD,M5,2144,strategy03_crypto\ETHUSD\M5\trades.csv
8,strategy03_crypto,LNKUSD,M5,1245,strategy03_crypto\LNKUSD\M5\trades.csv
9,strategy03_crypto,LTCUSD,M5,1725,strategy03_crypto\LTCUSD\M5\trades.csv


### Every crypto CSV path (relative to results root)

strategy03_crypto\ADAUSD\M5\merics.csv
strategy03_crypto\ADAUSD\M5\trades.csv
strategy03_crypto\AVEUSD\M5\merics.csv
strategy03_crypto\AVEUSD\M5\trades.csv
strategy03_crypto\AVXUSD\M5\merics.csv
strategy03_crypto\AVXUSD\M5\trades.csv
strategy03_crypto\BCHUSD\M5\merics.csv
strategy03_crypto\BCHUSD\M5\trades.csv
strategy03_crypto\BNBUSD\M5\merics.csv
strategy03_crypto\BNBUSD\M5\trades.csv
strategy03_crypto\BTCUSD\M5\merics.csv
strategy03_crypto\BTCUSD\M5\trades.csv
strategy03_crypto\DOGEUSD\M5\merics.csv
strategy03_crypto\DOGEUSD\M5\trades.csv
strategy03_crypto\ETHUSD\M5\merics.csv
strategy03_crypto\ETHUSD\M5\trades.csv
strategy03_crypto\LNKUSD\M5\merics.csv
strategy03_crypto\LNKUSD\M5\trades.csv
strategy03_crypto\LTCUSD\M5\merics.csv
strategy03_crypto\LTCUSD\M5\trades.csv
strategy03_crypto\SOLUSD\M5\merics.csv
strategy03_crypto\SOLUSD\M5\trades.csv
strategy03_crypto\TRXUSD\M5\merics.csv
strategy03_crypto\TRXUSD\M5\trades.csv
strategy03_crypto\UNIUSD\M5\merics.csv
strategy03_crypto\UNIUS